## Differential expression: treated vs control (Salmon + tximport + DESeq2)

Input: one Salmon `quant.sf` per sample, plus a sample sheet (CSV with columns `sample`, `condition`).
Output: gene-level counts table, PCA plot, DESeq2 results table with gene symbols, volcano plots.

Set every value in the **Parameters** cell; nothing else needs editing.

In [ ]:
library(DESeq2)
library(tximport)
library(ensembldb)
library(AnnotationHub)
library(org.Hs.eg.db)
library(EnhancedVolcano)
library(ggplot2)

Optional one-time installs (uncomment if a package is missing):

In [ ]:
# if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# BiocManager::install(c("DESeq2", "tximport", "ensembldb", "AnnotationHub",
#                        "org.Hs.eg.db", "EnhancedVolcano"))

## Parameters

In [ ]:
quants_dir   <- "Path_to_salmon_out"            # folder containing <sample>/quant.sf
sample_sheet <- "Path_to_sample_sheet.csv"      # columns: sample, condition
out_prefix   <- "deseq2_treated_vs_control"     # prefix for all output files

ref_level    <- "control"                       # denominator of the fold change
test_level   <- "treated"                       # numerator of the fold change

# Gene annotation: Ensembl release used to build the Salmon index
ensdb_species <- "sapiens"
ensdb_version <- "109"
ensdb_ah_id   <- NULL                           # optional AnnotationHub ID; NULL = look up by species/version

# Volcano plot
p_cutoff       <- 1e-4
fc_cutoff      <- 1
selected_genes <- c()                           # optional gene symbols to label, e.g. c("GENE_1", "GENE_2")

## 1. Transcript-to-gene map

In [ ]:
hub <- AnnotationHub()
if (is.null(ensdb_ah_id)) {
  q <- query(hub, c("EnsDb", ensdb_species, ensdb_version))
  print(q)
  stopifnot("expected exactly one matching EnsDb; set ensdb_ah_id" = length(q) == 1)
  ensdb_ah_id <- names(q)
}
ensdb <- hub[[ensdb_ah_id]]

tx_data <- transcripts(ensdb, return.type = "DataFrame")
tx2gene <- tx_data[, c("tx_id", "gene_id")]
head(tx2gene)

## 2. Import Salmon quantification

In [ ]:
coldata <- read.csv(sample_sheet, stringsAsFactors = FALSE)
stopifnot(all(c("sample", "condition") %in% colnames(coldata)))
rownames(coldata) <- coldata$sample

quant_files <- file.path(quants_dir, coldata$sample, "quant.sf")
names(quant_files) <- coldata$sample
stopifnot("missing quant.sf for some samples" = all(file.exists(quant_files)))
stopifnot(ref_level %in% coldata$condition, test_level %in% coldata$condition)

txi <- tximport(quant_files, type = "salmon", tx2gene = tx2gene, ignoreTxVersion = TRUE)

write.table(txi$counts, file = paste0(out_prefix, "_gene_counts.txt"),
            sep = "\t", row.names = TRUE, col.names = NA, quote = FALSE)
coldata

## 3. DESeq2

In [ ]:
coldata$condition <- factor(coldata$condition)
dds <- DESeqDataSetFromTximport(txi, coldata, ~condition)
dds$condition <- relevel(dds$condition, ref = ref_level)
dds <- DESeq(dds)
dds

PCA on variance-stabilised counts:

In [ ]:
vsdata <- vst(dds, blind = FALSE)
plotPCA(vsdata, intgroup = "condition")

Results. The order in `contrast` sets the direction: positive log2FoldChange = higher in `test_level`.
No p-value or expression filter is applied here (rows with NA are dropped); filter downstream if needed,
for example `padj < 0.05 & baseMean > 10`.

In [ ]:
res <- results(dds, contrast = c("condition", test_level, ref_level))
res <- na.omit(res)

res.df <- as.data.frame(res)
res.df$symbol <- mapIds(org.Hs.eg.db, keys = rownames(res.df),
                        keytype = "ENSEMBL", column = "SYMBOL")
write.csv(res.df, paste0(out_prefix, "_results_nofilter.csv"))
head(res.df)

## 4. Volcano plots

In [ ]:
EnhancedVolcano(res.df, x = "log2FoldChange", y = "padj", lab = res.df$symbol)

In [ ]:
EnhancedVolcano(res.df, x = "log2FoldChange", y = "padj", lab = res.df$symbol,
                pCutoff = p_cutoff, FCcutoff = fc_cutoff)

In [ ]:
if (length(selected_genes) > 0) {
  EnhancedVolcano(res.df, x = "log2FoldChange", y = "padj", lab = res.df$symbol,
                  pCutoff = p_cutoff, FCcutoff = fc_cutoff, selectLab = selected_genes)
}